# Churn monitoring: from EDA to AI Orchestration runs

Explore the synthetic churn data locally, then inspect models trained by `flows/main/flow.py` on Outerbounds. This notebook does not train or write artifacts. The flow is the source of truth for preprocessing, fitting, evaluation, and Cards.

Run this notebook with the `dev3.14` environment for `pn.ui`. The EDA and Panel sections run before any remote deployment. After the flow is deployed and triggered, start Jupyter with the platform's S3 datastore and metadata service to inspect Outerbounds results.

In [1]:
import sys
from pathlib import Path

import hvplot.pandas  # noqa: F401
import panel as pn

sys.path.insert(0, str(Path.cwd() / 'src'))
from flows.main.flow import make_churn_dataset

pn.extension(throttled=True)

## Explore the dataset

The flow generates this same fixed synthetic cohort. These columns resemble churn features but do not represent real customer data.

In [2]:
df = make_churn_dataset(random_state=42)
display(df.head())
display(df.describe(include='all'))
print(f"Overall churn rate: {df['churned'].mean():.1%}")

,customer_id,tenure_months,monthly_charges,total_charges,support_tickets,avg_session_minutes,contract_type,churned
0,CUST-100000,47,50.56,3567.34,9,58.7,one-year,0
1,CUST-100001,41,67.70,2776.33,6,54.8,one-year,0
2,CUST-100002,35,82.11,3281.87,8,49.3,one-year,0
3,CUST-100003,2,38.86,2863.55,3,38.3,two-year,0
4,CUST-100004,41,63.44,6046.80,7,40.9,two-year,0


,customer_id,tenure_months,monthly_charges,total_charges,support_tickets,avg_session_minutes,contract_type,churned
count,3000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000,3000.000000
unique,3000,NaN,NaN,NaN,NaN,NaN,3,NaN
top,CUST-100000,NaN,NaN,NaN,NaN,NaN,one-year,NaN
freq,1,NaN,NaN,NaN,NaN,NaN,2171,NaN
mean,NaN,38.065333,61.944980,3565.513897,6.200333,51.089033,NaN,0.275333
std,NaN,11.265613,12.843988,1210.485180,1.675451,10.012314,NaN,0.446757
min,NaN,0.000000,20.000000,50.000000,0.000000,2.000000,NaN,0.000000
25%,NaN,31.000000,53.270000,2656.715000,5.000000,45.600000,NaN,0.000000
50%,NaN,38.000000,61.970000,3515.000000,6.000000,51.700000,NaN,0.000000
75%,NaN,46.000000,70.507500,4395.267500,7.000000,57.600000,NaN,1.000000


Overall churn rate: 27.5%


In [3]:
churn_by_contract = df.groupby('contract_type')['churned'].mean().mul(100)
bars = churn_by_contract.hvplot.bar(
    color='#08ca4a', title="Churn rate by contract type",
    xlabel='Contract type', ylabel='Churn rate (%)',
    height=300, responsive=True,
)

hist = df['tenure_months'].hvplot.hist(
    bins=30, color='#08ca4a', title="Tenure distribution",
    xlabel='Tenure (months)', ylabel='Customers', height=300, responsive=True,
)

bars + hist

:Layout
   .Bars.Churned :Bars   [contract_type]   (churned)
   .Histogram.I  :Histogram   [tenure_months]   (Count)

## From notebook output to Panel components

Panel turns Python objects into browser components. `panel.ui`, available as `pn.ui`, contains Material-style components such as `pn.ui.DataFrame`, `pn.ui.Markdown`, `pn.ui.Select`, and `pn.ui.IntSlider`. Start by replacing a static notebook table with a component; no app server or model training is needed.

In [4]:
pn.ui.Column(
    pn.ui.Markdown('### A static view of the churn cohort'),
    pn.ui.DataFrame(df.head(10), index=False),
    pn.ui.HoloViews(bars)
)

Column(design=<class 'panel.ui.theme.Mat..., objects=[Markdown, DataFrame, ...])

### Reference binding

A widget has a value and a *reference* to that value: `contract.param.value`. Passing the reference to a component means Panel updates the component when the selection changes. Passing `contract.value` instead would capture only the current string. Change the selection and watch the text below it.

In [5]:
contract = pn.ui.Select(
    label='Contract type',
    options=sorted(df['contract_type'].unique().tolist()),
)
pn.ui.Column(contract, pn.ui.Markdown(contract.param.value), height=300)

Column(design=<class 'panel.ui.theme.Mat..., height=300, objects=[Select, Markdown])

### Compute with `pn.bind`

Reference binding copies a value into a component. When output needs a calculation, `pn.bind(function, widget)` calls the function with the widget's *current value* and calls it again whenever that value changes. Here both controls filter the EDA cohort, then update the table and churn rate without fitting a model.

In [6]:
contract = pn.ui.Select(
    label='Contract type',
    options=sorted(df['contract_type'].unique().tolist()),
)
min_tenure = pn.ui.IntSlider(
    label='Minimum tenure (months)', start=0, end=72, value=0
)

def select_customers(contract_type, tenure):
    return df[(df['contract_type'] == contract_type) & (df['tenure_months'] >= tenure)]

def churn_summary(contract_type, tenure):
    subset = select_customers(contract_type, tenure)
    return f'**{len(subset):,} customers**, churn rate **{subset.churned.mean():.1%}**'

customers = pn.bind(select_customers, contract, min_tenure)
summary = pn.bind(churn_summary, contract, min_tenure)
pn.ui.Column(contract, min_tenure, pn.ui.Markdown(summary), pn.ui.DataFrame(customers.rx().head(10), index=False))

Column(design=<class 'panel.ui.theme.Mat..., objects=[Select, IntSlider, ...])

### Lay out an app

`pn.ui.Column` stacks components, `pn.ui.Row` puts them beside each other, and `pn.ui.Page` provides a titled app with a sidebar and main area. Reuse the same widgets and bindings rather than rewriting the filtering logic. In a served script, call `.servable()` on the Page; here the Page is shown in the notebook. See `02_interactive.py` for the served version with model predictions.

In [7]:
controls = pn.ui.Column(contract, min_tenure)
results = pn.ui.Column(
    pn.ui.Markdown('### Selected customers'),
    pn.ui.Markdown(summary),
    pn.ui.DataFrame(customers.rx().head(20), index=False),
)
pn.ui.Row(controls, results)

Row(design=<class 'panel.ui.theme.Mat..., objects=[Column, Column])

In [8]:
page = pn.ui.Page(
    title='Churn cohort explorer',
    sidebar=[contract, min_tenure],
    main=[
        pn.ui.Column(
            pn.ui.Markdown(summary), pn.ui.DataFrame(customers, index=False)
        )
    ],
)
page.servable()
page.preview(width=1200, height=800)

HTML(str, design=<class 'panel.ui.theme.Mat..., height=800, sizing_mode='fixed', width=1200)